# Dimensionality Reduction & Clustering

Package-backed computational workflow for fitted gene-expression trajectories and dynamic-mode clustering. Publication plotting is intentionally kept out of this notebook for now.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import chronopack as cp


In [ ]:
def read_original_notebook_dir(experiment_dir):
    text = (Path(experiment_dir) / "OriginalDirectoryPath.txt").read_text().strip()
    return Path(text.replace("~", str(Path.home()))).resolve()

workspace_dir = Path.cwd().resolve()
if workspace_dir.name == "k562-tnfa-puromycin.hg38":
    repo_root = workspace_dir.parent
else:
    repo_root = workspace_dir

puro_dir = repo_root / "k562-tnfa-puromycin.hg38"
k562a_dir = repo_root / "k562-tnfa.hg38.rep"
lps_dir = repo_root / "thp1-k562-lps.hg38.rep"

root_puro = read_original_notebook_dir(puro_dir).parent
root_k562a = read_original_notebook_dir(k562a_dir).parent
root_lps = read_original_notebook_dir(lps_dir).parent

SELF_CONTAINED_OUTDIR = puro_dir / "self_contained_outputs" / "global_analysis_extended"
SELF_CONTAINED_OUTDIR.mkdir(parents=True, exist_ok=True)

{
    "root_puro": str(root_puro),
    "root_k562a": str(root_k562a),
    "root_lps": str(root_lps),
    "output": str(SELF_CONTAINED_OUTDIR),
}


## Characterize Fitted Gene Trajectories

This extracts fitted curves from fixed tradeSeq sidecars, aligns/rescales time grids, standardizes fitted trajectories, computes derivatives, and calculates trajectory descriptors.


In [ ]:
char = cp.tl.characterize_gene_trajectories(
    root_puro=root_puro,
    root_k562a=root_k562a,
    root_lps=root_lps,
    output_dir=SELF_CONTAINED_OUTDIR,
)

char.adata.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_characterization.h5ad")
{
    "adata_shape": char.adata.shape,
    "time_grid": char.metadata["time_grid"],
    "pc_df_characterized_rows": None if char.pc_df is None else char.pc_df.shape[0],
}


## Cluster Gene Trajectories

This preserves the current Ward.D2 signed clustering workflow with alpha-weighted trajectory and derivative distances. Native tradeSeq sidecars are retained upstream for plotting compatibility.


In [ ]:
cluster = cp.tl.cluster_gene_trajectories(
    characterization=char,
    output_dir=SELF_CONTAINED_OUTDIR,
)

cluster.adata.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_clusters.h5ad")
pc_df = cluster.pc_df
pc_df.to_csv(SELF_CONTAINED_OUTDIR / "pc_df.new.csv")

{
    "adata_shape": cluster.adata.shape,
    "pc_df_shape": pc_df.shape,
    "broad_cluster_counts": pc_df["broad_cluster"].astype(str).value_counts().to_dict(),
}


## Inspect Outputs


In [ ]:
display(pc_df.head())
print(pc_df.shape)
print(pc_df["broad_cluster"].value_counts())


## Optional Reference Validation


In [ ]:
import os

if os.environ.get("CHRONOPACK_VALIDATE_REFERENCE") == "1":
    reference_path = root_puro / "intmd" / "global_analysis" / "extended" / "pc_df.new.rds"
    print("Reference validation requested.")
    print("Reference path:", reference_path)
    print("Current rows:", pc_df.shape[0])
    print("Current columns:", pc_df.shape[1])
else:
    print("Set CHRONOPACK_VALIDATE_REFERENCE=1 to run external reference validation.")
